<div align="center">

<!-- MOTIONSALT branded banner. Rendered as HTML for the logo mark + gradient. -->
<div style="background:linear-gradient(135deg,#0f172a 0%,#1e1b4b 60%,#312e81 100%);padding:28px 24px;border-radius:14px;color:#f8fafc;font-family:-apple-system,Segoe UI,Roboto,sans-serif;">
  <div style="display:flex;align-items:center;justify-content:center;gap:14px;">
    <div style="width:44px;height:44px;border-radius:10px;background:linear-gradient(135deg,#22d3ee,#a855f7);display:flex;align-items:center;justify-content:center;font-weight:900;font-size:22px;color:#0f172a;">M</div>
    <div style="font-size:30px;font-weight:800;letter-spacing:2px;">MOTIONSALT</div>
  </div>
  <div style="margin-top:8px;font-size:14px;opacity:0.85;letter-spacing:3px;text-transform:uppercase;">Anime&nbsp;Video&nbsp;Upscaler</div>
  <div style="margin-top:14px;font-size:14px;opacity:0.75;max-width:640px;margin-left:auto;margin-right:auto;">A free, no-install, GPU-in-the-cloud alternative to Topaz Video AI. Powered by AnimeJaNai&nbsp;V3 and Real-ESRGAN AnimeVideo&nbsp;v3.</div>
  <div style="margin-top:18px;font-size:12px;opacity:0.7;">
    <a style="color:#a5f3fc;text-decoration:none;" href="https://github.com/motionssalt/upscale">github.com/motionssalt/upscale</a>
  </div>
</div>

</div>

---

**How this works:** step through the four cells below in order. Each cell is a self-contained step of a wizard — Connect ➜ Upload ➜ Configure ➜ Download. You never need to read or edit any code.

## Step 1 — Connect

Click **Connect** below. This verifies your GPU, installs the dependencies, and downloads the AI model weights from the MOTIONSALT GitHub Releases (never from HuggingFace — see the README for why).

In [ ]:
#@title 🔌 Step 1 — Connect { display-mode: "form" }
#@markdown Click the **Connect** button that appears below this cell after you run it.
import os, sys, subprocess, shutil, json, time, urllib.request, urllib.error
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# ---------- MOTIONSALT global state ----------
MS = globals().setdefault("MOTIONSALT", {})
MS.setdefault("workdir", Path("/content/motionsalt"))
MS["workdir"].mkdir(parents=True, exist_ok=True)
MS.setdefault("weights_dir", MS["workdir"] / "weights")
MS["weights_dir"].mkdir(parents=True, exist_ok=True)
MS.setdefault("connected", False)

# ---------- Config: where to pull weights from ----------
# The Colab notebook ONLY ever downloads weights from this GitHub repo's Releases.
# The HuggingFace upstream is mirrored by a scheduled Action; the notebook itself
# never contacts HuggingFace directly.
GH_REPO   = "motionssalt/upscale"          # <-- change to your fork
GH_TAG    = "latest"                                    # "latest" resolves to the newest weights-vX.Y.Z tag
WEIGHTS = {
    "LOW":    "2x_AnimeJaNaiV3_SuperUltraCompact.pth",
    "MEDIUM": "2x_AnimeJaNaiV3_UltraCompact.pth",
    "HIGH":   "realesr-animevideov3.pth",
}
MS["weights_map"] = WEIGHTS
MS["gh_repo"]     = GH_REPO

# ---------- Branded status log ----------
_log = widgets.HTML(value="")
_lines = []
def status(kind, msg):
    icon = {"ok":"✅","warn":"⚠️","err":"❌","run":"⏳","info":"•"}[kind]
    color = {"ok":"#16a34a","warn":"#d97706","err":"#dc2626","run":"#0891b2","info":"#475569"}[kind]
    _lines.append(f'<div style="font-family:ui-monospace,Menlo,monospace;font-size:12.5px;color:{color};padding:2px 0;">{icon}&nbsp;&nbsp;{msg}</div>')
    _log.value = "".join(_lines)

def section(title):
    _lines.append(f'<div style="margin:10px 0 4px 0;font-weight:600;color:#0f172a;font-family:-apple-system,Segoe UI,sans-serif;">{title}</div>')
    _log.value = "".join(_lines)

# ---------- The Connect button ----------
btn = widgets.Button(
    description="Connect",
    icon="plug",
    button_style="primary",
    layout=widgets.Layout(width="180px", height="42px"),
)
badge = widgets.HTML(
    '<span style="display:inline-block;padding:4px 10px;border-radius:999px;background:#e2e8f0;color:#334155;font-size:11px;font-family:-apple-system,sans-serif;">not connected</span>'
)
header = widgets.HTML(
    '<div style="font-family:-apple-system,Segoe UI,sans-serif;font-weight:700;font-size:16px;color:#0f172a;">Connect to a MOTIONSALT session</div>'
    '<div style="font-family:-apple-system,Segoe UI,sans-serif;font-size:12.5px;color:#475569;margin-bottom:8px;">Verifies GPU · installs dependencies · pulls model weights from GitHub Releases</div>'
)

def _run(cmd, quiet=True):
    """Run a shell command; return (returncode, tail_of_output)."""
    p = subprocess.run(cmd, shell=isinstance(cmd,str), capture_output=True, text=True)
    if not quiet and p.returncode != 0:
        print(p.stdout[-2000:]); print(p.stderr[-2000:])
    return p.returncode, (p.stderr or p.stdout)[-400:]

def _resolve_release_tag():
    """Ask GitHub for the newest release tag (unauthenticated is fine — public repo)."""
    if GH_TAG != "latest":
        return GH_TAG
    url = f"https://api.github.com/repos/{GH_REPO}/releases/latest"
    with urllib.request.urlopen(url, timeout=20) as r:
        data = json.loads(r.read().decode("utf-8"))
    return data["tag_name"]

def _download(url, dest: Path, label: str):
    """Streaming download with a live progress bar."""
    bar = widgets.IntProgress(value=0, min=0, max=100, description=label,
                              layout=widgets.Layout(width="100%"),
                              bar_style="info")
    pct = widgets.HTML(value="0%")
    row = widgets.HBox([bar, pct])
    display(row)
    req = urllib.request.Request(url, headers={"User-Agent":"motionsalt-upscaler"})
    with urllib.request.urlopen(req, timeout=60) as r:
        total = int(r.headers.get("Content-Length", "0")) or 0
        read = 0
        with dest.open("wb") as f:
            while True:
                chunk = r.read(1 << 20)
                if not chunk:
                    break
                f.write(chunk); read += len(chunk)
                if total:
                    p = int(read * 100 / total)
                    bar.value = p; pct.value = f"{p}%"
        if total:
            bar.value = 100; pct.value = "100%"
        bar.bar_style = "success"

def on_connect(_):
    btn.disabled = True
    badge.value = '<span style="display:inline-block;padding:4px 10px;border-radius:999px;background:#fef3c7;color:#92400e;font-size:11px;">connecting…</span>'
    _lines.clear(); _log.value = ""

    # 1. GPU
    section("1 / 4 · GPU")
    try:
        import torch
        if not torch.cuda.is_available():
            status("err", "No CUDA GPU detected. Enable GPU: Runtime → Change runtime type → GPU.")
            badge.value = '<span style="padding:4px 10px;border-radius:999px;background:#fee2e2;color:#991b1b;font-size:11px;">no GPU</span>'
            btn.disabled = False
            return
        name = torch.cuda.get_device_name(0)
        status("ok", f"GPU detected: <b>{name}</b>")
    except Exception as e:
        status("err", f"PyTorch not importable yet: {e}")

    # 2. System deps (ffmpeg)
    section("2 / 4 · System dependencies")
    if shutil.which("ffmpeg"):
        status("ok", "ffmpeg already present.")
    else:
        status("run", "Installing ffmpeg…")
        rc, tail = _run("apt-get -qq update && apt-get -qq install -y ffmpeg")
        status("ok" if rc==0 else "err", "ffmpeg installed." if rc==0 else f"ffmpeg install failed: {tail}")

    # 3. Python deps
    section("3 / 4 · Python packages")
    pkgs = ["opencv-python-headless", "numpy", "spandrel", "tqdm"]
    # spandrel: universal loader that supports both AnimeJaNai (compact/SRVGGNet) and Real-ESRGAN (SRVGGNet) checkpoints.
    status("run", "Installing " + ", ".join(pkgs) + " …")
    rc, tail = _run([sys.executable, "-m", "pip", "install", "-q", *pkgs])
    status("ok" if rc==0 else "err", "Python packages ready." if rc==0 else f"pip failed: {tail}")

    # 4. Weights — from THIS GitHub repo, never HuggingFace
    section("4 / 4 · Model weights (from GitHub Releases)")
    try:
        tag = _resolve_release_tag()
        status("ok", f"Resolved release tag: <b>{tag}</b>")
    except Exception as e:
        status("err", f"Could not reach GitHub API: {e}")
        btn.disabled = False; return

    ok_all = True
    for tier, fname in WEIGHTS.items():
        dest = MS["weights_dir"] / fname
        if dest.exists() and dest.stat().st_size > 0:
            status("ok", f"{tier} — {fname} already cached.")
            continue
        url = f"https://github.com/{GH_REPO}/releases/download/{tag}/{fname}"
        status("run", f"{tier} — downloading {fname}…")
        try:
            _download(url, dest, tier)
            status("ok", f"{tier} — downloaded.")
        except Exception as e:
            status("err", f"{tier} — download failed: {e}")
            ok_all = False

    if ok_all:
        MS["connected"] = True
        MS["release_tag"] = tag
        badge.value = '<span style="padding:4px 10px;border-radius:999px;background:#dcfce7;color:#166534;font-size:11px;">connected</span>'
        status("ok", "<b>Ready.</b> Continue to Step 2.")
    else:
        badge.value = '<span style="padding:4px 10px;border-radius:999px;background:#fee2e2;color:#991b1b;font-size:11px;">error</span>'

    btn.disabled = False

btn.on_click(on_connect)

panel = widgets.VBox([
    header,
    widgets.HBox([btn, badge]),
    _log,
], layout=widgets.Layout(border="1px solid #e2e8f0", padding="14px", border_radius="10px"))
display(panel)


## Step 2 — Upload your video

Pick a video file from your device. It's copied into the Colab VM only — nothing is sent to a third-party service.

In [ ]:
#@title 📤 Step 2 — Upload video { display-mode: "form" }
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from pathlib import Path
import shutil, os

MS = globals().setdefault("MOTIONSALT", {})
if not MS.get("connected"):
    display(HTML('<div style="padding:10px;background:#fee2e2;color:#991b1b;border-radius:8px;font-family:sans-serif;">⚠️ Run Step 1 first (Connect).</div>'))
else:
    header = widgets.HTML(
        '<div style="font-family:-apple-system,Segoe UI,sans-serif;font-weight:700;font-size:16px;color:#0f172a;">Upload the video you want to upscale</div>'
        '<div style="font-family:-apple-system,Segoe UI,sans-serif;font-size:12.5px;color:#475569;margin-bottom:10px;">MP4, MKV, MOV, WebM, AVI… anything ffmpeg reads.</div>'
    )
    pick_btn = widgets.Button(description="Choose file…", icon="upload",
                              button_style="primary",
                              layout=widgets.Layout(width="180px", height="42px"))
    prog = widgets.IntProgress(value=0, min=0, max=100, description="Copy",
                               layout=widgets.Layout(width="100%"),
                               bar_style="info")
    prog_pct = widgets.HTML("0%")
    prog_row = widgets.HBox([prog, prog_pct])
    prog_row.layout.display = "none"
    result = widgets.HTML("")

    def on_pick(_):
        # Use the Colab file uploader; then copy into workspace with progress.
        from google.colab import files
        pick_btn.disabled = True
        result.value = '<div style="font-family:sans-serif;font-size:12.5px;color:#475569;">Waiting for browser file picker…</div>'
        uploaded = files.upload()
        if not uploaded:
            result.value = '<div style="padding:8px;background:#fef3c7;color:#92400e;border-radius:6px;font-family:sans-serif;">No file selected.</div>'
            pick_btn.disabled = False
            return
        name, data = next(iter(uploaded.items()))
        src_tmp = Path("/content") / name
        # google.colab.files.upload() already wrote it into /content; we just move it.
        dest = MS["workdir"] / "input" / name
        dest.parent.mkdir(parents=True, exist_ok=True)
        prog_row.layout.display = "flex"
        # Copy with progress (chunked, so the bar is real, not fake).
        total = len(data); read = 0
        with open(src_tmp, "rb") as fin, open(dest, "wb") as fout:
            while True:
                chunk = fin.read(1 << 20)
                if not chunk: break
                fout.write(chunk); read += len(chunk)
                prog.value = int(read * 100 / max(total, 1))
                prog_pct.value = f"{prog.value}%"
        prog.value = 100; prog_pct.value = "100%"; prog.bar_style = "success"
        try: os.remove(src_tmp)
        except OSError: pass
        MS["input_path"] = dest
        size_mb = dest.stat().st_size / 1e6
        result.value = (
            f'<div style="padding:10px;background:#dcfce7;color:#166534;border-radius:8px;font-family:sans-serif;">'
            f'✅ Uploaded <b>{name}</b> ({size_mb:.1f} MB). Continue to Step 3.'
            f'</div>'
        )
        pick_btn.disabled = False

    pick_btn.on_click(on_pick)
    display(widgets.VBox([header, pick_btn, prog_row, result],
                         layout=widgets.Layout(border="1px solid #e2e8f0", padding="14px", border_radius="10px")))


## Step 3 — Configure & process

Pick a quality tier and dial in the filters. Then click **Start Processing**. A progress bar shows real frame-by-frame progress.

In [ ]:
#@title ⚙️ Step 3 — Configure & process { display-mode: "form" }
import ipywidgets as widgets
from IPython.display import display, HTML
from pathlib import Path
import subprocess, shutil, math, json, time, os, sys, threading, collections, queue

MS = globals().setdefault("MOTIONSALT", {})
if not MS.get("connected"):
    display(HTML('<div style="padding:10px;background:#fee2e2;color:#991b1b;border-radius:8px;font-family:sans-serif;">⚠️ Run Step 1 first (Connect).</div>'))
elif not MS.get("input_path"):
    display(HTML('<div style="padding:10px;background:#fee2e2;color:#991b1b;border-radius:8px;font-family:sans-serif;">⚠️ Run Step 2 first (Upload).</div>'))
else:
    # -------- Widgets --------
    # NOTE (Bug 0): the model dropdown intentionally does NOT print a fixed
    # "N×" in its labels. Native scale is read off the loaded model at runtime
    # (spandrel's ImageModelDescriptor exposes .scale), so nothing hardcodes 2×.
    tier = widgets.Dropdown(
        options=[
            ("LOW  — AnimeJaNai V3 SuperUltraCompact (fastest)", "LOW"),
            ("MEDIUM — AnimeJaNai V3 UltraCompact (balanced)", "MEDIUM"),
            ("HIGH — Real-ESRGAN AnimeVideo v3 (best quality)", "HIGH"),
        ],
        value="MEDIUM",
        description="Quality",
        style={"description_width": "160px"},
        layout=widgets.Layout(width="640px"),
    )

    def slider(desc, default=0):
        return widgets.IntSlider(
            value=default, min=0, max=100, step=1, description=desc,
            style={"description_width": "160px"},
            layout=widgets.Layout(width="640px"),
            continuous_update=False,
        )

    s_revert    = slider("Revert Compression", 0)
    s_detail    = slider("Improve Detail", 0)
    s_sharpen   = slider("Sharpen", 15)
    s_denoise   = slider("Reduce Noise", 0)
    s_dehalo    = slider("Dehalo", 0)
    s_deblur    = slider("Anti-alias/Deblur", 0)
    s_recover   = slider("Recover Original Detail", 0)
    cb_1080     = widgets.Checkbox(value=False, description="Downscale to 1080p height (preserve aspect ratio)",
                                   indent=False)
    # BUGFIX (this pass): fp16 toggle removed by user request. Previous debug
    # rounds kept flipping fp16 on/off as a "just in case" lever and it made
    # zero measurable difference to end-to-end fps — the bottleneck was never
    # the model's math precision. Everything below runs in fp32. If a future
    # pass wants to reintroduce half precision, do it behind a Boolean that is
    # explicitly validated against the profiler output, not a UI toggle.

    start = widgets.Button(description="Start Processing", icon="play",
                           button_style="success",
                           layout=widgets.Layout(width="220px", height="44px"))
    frame_prog = widgets.IntProgress(value=0, min=0, max=100, description="Frames",
                                     layout=widgets.Layout(width="100%"), bar_style="info")
    frame_pct  = widgets.HTML("0%")
    stage_lbl  = widgets.HTML("")
    log        = widgets.HTML("")
    _msgs = []
    def logline(kind, msg):
        icon = {"ok":"✅","warn":"⚠️","err":"❌","run":"⏳","info":"•","perf":"📊"}[kind]
        color = {"ok":"#16a34a","warn":"#d97706","err":"#dc2626","run":"#0891b2","info":"#475569","perf":"#7c3aed"}[kind]
        _msgs.append(f'<div style="font-family:ui-monospace,Menlo,monospace;font-size:12.5px;color:{color};padding:2px 0;">{icon}&nbsp;&nbsp;{msg}</div>')
        log.value = "".join(_msgs)
        # Also mirror perf/info lines to stdout so they survive kernel restarts
        # and can be grepped out of the notebook JSON afterwards.
        if kind in ("perf", "warn", "err", "ok"):
            print(f"[motionsalt/{kind}] {msg}", flush=True)

    hdr = widgets.HTML(
        '<div style="font-family:-apple-system,Segoe UI,sans-serif;font-weight:700;font-size:16px;color:#0f172a;">Configure processing</div>'
        '<div style="font-family:-apple-system,Segoe UI,sans-serif;font-size:12.5px;color:#475569;margin-bottom:10px;">Defaults are safe. All sliders are 0–100.</div>'
    )

    def start_processing(_):
        start.disabled = True
        _msgs.clear(); log.value = ""
        try:
            _do_process()
        except Exception as e:
            logline("err", f"Processing failed: {e}")
            raise
        finally:
            start.disabled = False

    # ---------- The actual pipeline ----------
    def _ffprobe(path, args):
        out = subprocess.check_output(["ffprobe","-v","error", *args, str(path)]).decode().strip()
        return out

    def _nvidia_smi_snapshot():
        """One-shot GPU util + mem snapshot via nvidia-smi. Returns a short
        string like 'gpu=87% mem=3421/15109MiB' or '' if nvidia-smi is missing.
        Cheap enough to call every few seconds; we do NOT call it in the hot
        loop. Used to prove-or-disprove the 'GPU is idle' hypothesis without
        making assumptions."""
        try:
            out = subprocess.check_output(
                ["nvidia-smi",
                 "--query-gpu=utilization.gpu,memory.used,memory.total",
                 "--format=csv,noheader,nounits"],
                stderr=subprocess.DEVNULL, timeout=1.5,
            ).decode().strip().splitlines()[0]
            util, used, total = [x.strip() for x in out.split(",")]
            return f"gpu={util}% mem={used}/{total}MiB"
        except Exception:
            return ""

    def _load_model(tier_val):
        """Load the checkpoint for the chosen tier via spandrel. Always fp32
        (the fp16 lever was removed — see cb_fp16 note above).

        Returns (descriptor, scale). We read `.scale` off the loaded
        ImageModelDescriptor so nothing downstream has to hardcode a factor.
        """
        import torch
        from spandrel import ModelLoader, ImageModelDescriptor
        weight_file = MS["weights_map"][tier_val]
        path = MS["weights_dir"] / weight_file
        model = ModelLoader().load_from_file(str(path))
        if not isinstance(model, ImageModelDescriptor):
            raise RuntimeError(
                f"{weight_file} did not load as an ImageModelDescriptor "
                f"(got {type(model).__name__}) — cannot use this checkpoint.")
        model.cuda().eval()
        scale = int(getattr(model, "scale", 0)) or 0
        if scale <= 0:
            raise RuntimeError(
                f"Loaded {weight_file} but could not determine its upscale "
                f"factor from the descriptor (.scale={scale!r}). Refusing to "
                f"guess — that is exactly the assumption that broke before.")
        return model, scale

    def _apply_pre_filters(bgr, params):
        """Filters that logically belong BEFORE the model: deblock + denoise.
        Cheap on the small source; would be an order of magnitude slower on
        the upscaled output."""
        import cv2
        out = bgr
        if params["revert"] > 0:
            k = params["revert"] / 100.0
            d = int(3 + 6 * k)
            sC = 20 + 60 * k; sS = 20 + 40 * k
            out = cv2.bilateralFilter(out, d, sC, sS)
        if params["denoise"] > 0:
            h = 3 + 12 * (params["denoise"] / 100.0)
            out = cv2.fastNlMeansDenoisingColored(out, None, h, h, 7, 21)
        return out

    def _apply_post_filters(bgr, params):
        """Filters that only make sense on the upscaled output."""
        import cv2, numpy as np
        out = bgr
        if params["dehalo"] > 0:
            k = params["dehalo"] / 100.0
            gray = cv2.cvtColor(out, cv2.COLOR_BGR2GRAY)
            edges = cv2.Canny(gray, 60, 180)
            band  = cv2.dilate(edges, np.ones((3,3), np.uint8), iterations=1)
            band  = cv2.subtract(band, edges)
            med   = cv2.medianBlur(out, 5)
            mask  = (band > 0)[..., None].astype(np.float32) * (0.35 + 0.55 * k)
            out   = (out.astype(np.float32) * (1 - mask) + med.astype(np.float32) * mask).astype(np.uint8)
        if params["deblur"] > 0:
            k = params["deblur"] / 100.0
            sig = 0.4 + 0.9 * k
            blur = cv2.GaussianBlur(out, (0,0), sig)
            out  = cv2.addWeighted(out, 1.0 + 0.35 * k, blur, -0.35 * k, 0)
        if params["detail"] > 0:
            k = params["detail"] / 100.0
            lab = cv2.cvtColor(out, cv2.COLOR_BGR2LAB)
            L, A, B = cv2.split(lab)
            clahe = cv2.createCLAHE(clipLimit=1.0 + 2.5 * k, tileGridSize=(8,8))
            L = clahe.apply(L)
            out = cv2.cvtColor(cv2.merge([L,A,B]), cv2.COLOR_LAB2BGR)
        if params["sharpen"] > 0:
            k = params["sharpen"] / 100.0
            blur = cv2.GaussianBlur(out, (0,0), 1.2)
            amt  = 0.2 + 1.2 * k
            out  = cv2.addWeighted(out, 1.0 + amt, blur, -amt, 0)
        return out

    def _recover_blend(upscaled_bgr, original_bgr, strength_0_100):
        if strength_0_100 <= 0:
            return upscaled_bgr
        import cv2, numpy as np
        h, w = upscaled_bgr.shape[:2]
        naive = cv2.resize(original_bgr, (w, h), interpolation=cv2.INTER_LANCZOS4)
        alpha = 0.5 * (strength_0_100 / 100.0)
        return (upscaled_bgr.astype(np.float32) * (1 - alpha) + naive.astype(np.float32) * alpha).astype(np.uint8)

    # ------------------------------------------------------------------
    # PROFILING (this pass, Step 1 of the debug prompt)
    # ------------------------------------------------------------------
    # The previous code already had a per-stage `stage_times` dict, but it
    # only printed AFTER 10 frames. At ~0.2 fps that's a 50-second wait
    # before you learn anything. Worse, it lumped all model-side work into
    # one "infer" bucket, so you could not tell CPU-side stalls (H2D copy,
    # numpy strides, pinned-memory misses, Python overhead) from actual
    # kernel time — which is EXACTLY the distinction we need to answer
    # "is the GPU maxed out or idle?".
    #
    # This pass rewrites the profiling so:
    #   1. The `infer` bucket is split into `infer_h2d`, `infer_kernel` (real
    #      GPU wall-time via torch.cuda.Event), `infer_d2h`.
    #   2. Numbers are logged after 3 frames AND then every 5 frames for the
    #      first 20 frames — so a 0.2-fps run surfaces its breakdown after
    #      ~15 seconds, not ~50.
    #   3. A background heartbeat thread snapshots `nvidia-smi` GPU util
    #      every 3 s and logs it. This directly answers the user's belief
    #      that "GPU isn't being maxed out".
    #   4. Model device is asserted AND logged (Step 2 of the debug prompt).
    # ------------------------------------------------------------------

    def _infer_tensor(model, bgr, prof):
        """Run one BGR uint8 frame through the model; return BGR uint8.

        `prof` is a dict updated with:
            infer_h2d        — CPU-side time from numpy in to torch on GPU
            infer_kernel     — GPU wall-time of the forward pass (cuda.Event)
            infer_d2h        — GPU-to-host copy + numpy conversion
        These are broken out so we can tell CPU-bound overhead from real
        kernel work. Adds one CUDA stream-sync per call — negligible on the
        scale of a full forward pass, essential for meaningful timing.

        Note on non_blocking / pinned memory (retained investigation):
        `torch.from_numpy(...)` returns a tensor over PAGEABLE host memory.
        `.to("cuda", non_blocking=True)` on pageable memory is silently
        synchronous — the non_blocking hint is a no-op. To actually overlap
        H2D with compute you must pin first. We do NOT pin here (single-
        threaded loop — nothing to overlap with anyway), but flagging it
        so the next optimization pass knows the flag is currently cosmetic.
        """
        import torch, numpy as np

        # ---- H2D + preprocess (measured on CPU wall-time) ----
        t_h2d_0 = time.perf_counter()
        rgb = np.ascontiguousarray(bgr[:, :, ::-1])
        t = (torch.from_numpy(rgb)
             .permute(2, 0, 1)
             .unsqueeze(0)
             .to(device="cuda", dtype=torch.float32, non_blocking=True)
             .div(255.0))
        # Force sync so H2D time is measured honestly. Without this the
        # transfer time bleeds into the next stage's timer.
        torch.cuda.synchronize()
        prof["infer_h2d"] += time.perf_counter() - t_h2d_0

        # ---- Kernel forward pass (measured with CUDA events) ----
        ev_start = torch.cuda.Event(enable_timing=True)
        ev_end   = torch.cuda.Event(enable_timing=True)
        ev_start.record()
        with torch.no_grad():
            y = model(t)
        ev_end.record()
        torch.cuda.synchronize()
        prof["infer_kernel"] += ev_start.elapsed_time(ev_end) / 1000.0  # ms -> s

        # ---- D2H + postprocess (CPU wall-time) ----
        t_d2h_0 = time.perf_counter()
        y = (y.clamp(0.0, 1.0)
              .squeeze(0)
              .permute(1, 2, 0)
              .mul(255.0)
              .round()
              .to(torch.uint8))
        arr = y.contiguous().cpu().numpy()
        out = np.ascontiguousarray(arr[:, :, ::-1])
        prof["infer_d2h"] += time.perf_counter() - t_d2h_0
        return out

    def _drain_stderr(pipe, buf):
        try:
            for line in iter(pipe.readline, b""):
                try:
                    buf.append(line.decode("utf-8", errors="replace").rstrip())
                except Exception:
                    buf.append(repr(line))
        except Exception:
            pass
        finally:
            try: pipe.close()
            except Exception: pass

    def _gpu_watcher(stop_evt, samples):
        """Background thread: snapshot nvidia-smi every 3s and stash a small
        rolling window. Cheap (~30ms per call), external process — will not
        distort the profile of the main loop. This is the ONLY way to
        confidently say 'GPU utilization is 5%' vs 'GPU is saturated',
        which is the exact question the debug prompt asks in Step 2."""
        while not stop_evt.is_set():
            s = _nvidia_smi_snapshot()
            if s:
                samples.append((time.time(), s))
            # 3-second cadence: enough to catch a stall, sparse enough not
            # to matter. `stop_evt.wait()` returns True immediately when set,
            # so shutdown is prompt.
            if stop_evt.wait(3.0):
                return

    def _do_process():
        import cv2, numpy as np, torch
        in_path = MS["input_path"]
        out_dir = MS["workdir"] / "out"; out_dir.mkdir(exist_ok=True)
        stem = in_path.stem

        # 1. Probe input
        stage_lbl.value = "<b>Reading source metadata…</b>"
        fps = _ffprobe(in_path, ["-select_streams","v:0","-show_entries","stream=r_frame_rate","-of","csv=p=0"])
        num, den = (fps.split("/") + ["1"])[:2]
        fps_f = float(num) / float(den) if float(den) else 30.0
        nframes = int(_ffprobe(in_path, ["-select_streams","v:0","-count_packets","-show_entries","stream=nb_read_packets","-of","csv=p=0"]) or "0")
        logline("ok", f"Source: {fps_f:.3f} fps · ~{nframes or 'unknown'} frames.")

        # 2. Load model + GPU sanity check (Step 2 of the debug prompt).
        stage_lbl.value = "<b>Loading model…</b>"
        logline("run", f"Loading {tier.value} model…")
        cuda_ok = torch.cuda.is_available()
        logline("info", f"torch.cuda.is_available() = {cuda_ok} · "
                        f"torch={torch.__version__} · "
                        f"device_count={torch.cuda.device_count() if cuda_ok else 0}")
        if not cuda_ok:
            raise RuntimeError("CUDA is not available — refusing to run on CPU. "
                               "Reconnect to a GPU runtime and re-run Step 1.")

        # cudnn.benchmark: pick fastest conv algo for the (fixed) input shape.
        # Kept from previous pass. Confirmed relevant to Compact SRVGGNet.
        torch.backends.cudnn.benchmark = True

        model, scale = _load_model(tier.value)
        gpu_name = torch.cuda.get_device_name(0)
        logline("ok", f"{tier.value} model loaded on {gpu_name} · native scale {scale}× · fp32.")

        # Ground truth for "is it actually on GPU?" — read the device off the
        # first parameter, not off what we THINK .cuda() did. This is exactly
        # what the debug prompt asks for in Step 2.
        try:
            first_param = next(model.model.parameters()) if hasattr(model, "model") else next(model.parameters())
        except Exception:
            first_param = None
        if first_param is None:
            raise RuntimeError("Could not read model parameters to confirm device placement.")
        logline("info", f"First model param device = {first_param.device} · dtype = {first_param.dtype}")
        if first_param.device.type != "cuda":
            raise RuntimeError(
                f"Model parameters ended up on {first_param.device} instead of "
                f"cuda after .cuda() — refusing to run (would be ~0.02 fps).")

        # One-shot pre-run GPU snapshot so we have a "before" baseline.
        pre_snap = _nvidia_smi_snapshot()
        if pre_snap:
            logline("info", f"nvidia-smi (pre-run): {pre_snap}")

        params = dict(
            revert=s_revert.value, detail=s_detail.value, sharpen=s_sharpen.value,
            denoise=s_denoise.value, dehalo=s_dehalo.value, deblur=s_deblur.value,
            recover=s_recover.value,
        )

        # 3. Open source + set up ffmpeg pipe.
        cap = cv2.VideoCapture(str(in_path))
        if not cap.isOpened():
            raise RuntimeError("OpenCV could not open the input video.")
        w  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        if w <= 0 or h <= 0:
            raise RuntimeError(f"OpenCV reported invalid source dimensions {w}x{h}.")

        # Output dims from the model's actual declared scale.
        out_w, out_h = w * scale, h * scale
        # x264 yuv420p needs even dims (see Bug 1 note in previous pass).
        crop_w = w - (out_w % 2 != 0)
        crop_h = h - (out_h % 2 != 0)
        if (crop_w, crop_h) != (w, h):
            logline("warn",
                    f"Source {w}x{h} at scale {scale}× would produce odd "
                    f"output ({out_w}x{out_h}); cropping source to "
                    f"{crop_w}x{crop_h} to keep yuv420p happy.")
            w, h = crop_w, crop_h
            out_w, out_h = w * scale, h * scale
        if out_w % 2 or out_h % 2:
            raise RuntimeError(f"Refusing to write odd output dims {out_w}x{out_h}.")

        stage_lbl.value = (f"<b>Upscaling {w}×{h} → {out_w}×{out_h} ({scale}×, fp32) …</b>")

        video_only = out_dir / f"{stem}_upscaled_noaudio.mp4"

        ffmpeg_cmd = [
            "ffmpeg", "-y", "-hide_banner", "-loglevel", "error",
            "-f", "rawvideo",
            "-vcodec", "rawvideo",
            "-pix_fmt", "bgr24",
            "-s", f"{out_w}x{out_h}",
            "-r", f"{fps_f}",
            "-an",
            "-i", "-",
            "-c:v", "libx264",
            "-preset", "medium",
            "-crf", "16",
            "-pix_fmt", "yuv420p",
            "-movflags", "+faststart",
            str(video_only),
        ]
        logline("info", "ffmpeg: " + " ".join(ffmpeg_cmd))

        ff = subprocess.Popen(
            ffmpeg_cmd,
            stdin=subprocess.PIPE,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.PIPE,
            bufsize=0,
        )
        stderr_buf = collections.deque(maxlen=200)
        stderr_thread = threading.Thread(
            target=_drain_stderr, args=(ff.stderr, stderr_buf), daemon=True,
        )
        stderr_thread.start()

        time.sleep(0.25)
        if ff.poll() is not None:
            stderr_thread.join(timeout=1.0)
            tail = "\n".join(stderr_buf) or "(no stderr captured)"
            raise RuntimeError(
                f"ffmpeg exited immediately (returncode={ff.returncode}). "
                f"stderr:\n{tail}"
            )

        expected_frame_bytes = out_w * out_h * 3

        frame_prog.max = max(nframes, 1)
        i = 0; t0 = time.time()

        # ---- Profiling state ----
        # Timings are ACCUMULATED in seconds, then divided by frame count when
        # reported. Split into fine-grained buckets so we can point at the
        # actual bottleneck instead of guessing.
        stage_times = collections.defaultdict(float)
        # Report cadence: initial burst so slow runs surface numbers fast.
        report_at = {3, 5, 10, 15, 20, 30, 50, 100}
        buckets_ordered = ("read", "pre", "infer_h2d", "infer_kernel",
                           "infer_d2h", "recover", "post", "write")

        # ---- Background GPU util watcher ----
        gpu_samples = collections.deque(maxlen=200)
        gpu_stop = threading.Event()
        gpu_thread = threading.Thread(
            target=_gpu_watcher, args=(gpu_stop, gpu_samples), daemon=True,
        )
        gpu_thread.start()

        last_ui_update = 0.0
        last_stdout_beat = 0.0
        last_gpu_log = 0.0

        try:
            while True:
                s = time.perf_counter()
                ok, frame = cap.read()
                if not ok:
                    break
                if frame.shape[1] != w or frame.shape[0] != h:
                    frame = frame[:h, :w]
                stage_times["read"] += time.perf_counter() - s

                s = time.perf_counter()
                pre = _apply_pre_filters(frame, params) if (params["revert"] or params["denoise"]) else frame
                stage_times["pre"] += time.perf_counter() - s

                # Inference — updates the h2d/kernel/d2h buckets internally.
                up = _infer_tensor(model, pre, stage_times)

                s = time.perf_counter()
                if params["recover"] > 0:
                    up = _recover_blend(up, frame, params["recover"])
                stage_times["recover"] += time.perf_counter() - s

                s = time.perf_counter()
                if params["dehalo"] or params["deblur"] or params["detail"] or params["sharpen"]:
                    up = _apply_post_filters(up, params)
                stage_times["post"] += time.perf_counter() - s

                # Shape / byte sanity — unchanged from previous pass.
                if up.shape[0] != out_h or up.shape[1] != out_w or up.shape[2] != 3:
                    raise RuntimeError(
                        f"Frame {i}: post-processed shape {up.shape} does not "
                        f"match expected {out_h}x{out_w}x3 (source {h}x{w}, "
                        f"model native scale {scale}×).")
                buf = up.tobytes()
                if len(buf) != expected_frame_bytes:
                    raise RuntimeError(
                        f"Frame {i}: byte length {len(buf)} != expected "
                        f"{expected_frame_bytes}.")

                s = time.perf_counter()
                try:
                    ff.stdin.write(buf)
                except BrokenPipeError:
                    ff.wait(timeout=2.0)
                    stderr_thread.join(timeout=1.0)
                    tail = "\n".join(stderr_buf) or "(no stderr captured)"
                    raise RuntimeError(
                        f"Broken pipe while writing frame {i} to ffmpeg "
                        f"(returncode={ff.returncode}). stderr:\n{tail}"
                    )
                stage_times["write"] += time.perf_counter() - s
                i += 1

                # ---- Per-frame profile reports ----
                if i in report_at:
                    total = sum(stage_times[k] for k in buckets_ordered) or 1e-9
                    parts = []
                    for k in buckets_ordered:
                        v = stage_times[k]
                        parts.append(f"{k}={v/i*1000:.1f}ms ({v/total*100:.0f}%)")
                    fps_now = i / max(time.time() - t0, 1e-6)
                    logline("perf",
                            f"[frame {i}] {fps_now:.3f} fps · " + " · ".join(parts))

                # ---- Periodic GPU util log ----
                # Independent of frame count so it fires even at 0.05 fps.
                now = time.time()
                if (now - last_gpu_log) >= 6.0 and gpu_samples:
                    _, snap = gpu_samples[-1]
                    logline("perf", f"nvidia-smi: {snap} @ frame {i}")
                    last_gpu_log = now

                # ---- UI heartbeat ----
                if i == 1 or (now - last_ui_update) >= 0.5 or i == nframes:
                    frame_prog.value = min(i, frame_prog.max)
                    elapsed = now - t0
                    fps_now = i / max(elapsed, 1e-6)
                    eta = (nframes - i) / fps_now if (nframes and fps_now > 0) else 0
                    eta_str = f" · ETA {int(eta//60)}m{int(eta%60):02d}s" if eta else ""
                    frame_pct.value = f"{i}/{nframes or '?'} · {fps_now:.2f} fps{eta_str}"
                    last_ui_update = now
                if (now - last_stdout_beat) >= 10.0:
                    print(f"[motionsalt] frame {i}/{nframes or '?'} "
                          f"({i/max(now-t0,1e-6):.2f} fps)", flush=True)
                    last_stdout_beat = now
        finally:
            cap.release()
            try:
                if ff.stdin and not ff.stdin.closed:
                    ff.stdin.close()
            except Exception:
                pass
            gpu_stop.set()

        rc = ff.wait()
        stderr_thread.join(timeout=2.0)
        gpu_thread.join(timeout=4.0)
        if rc != 0:
            tail = "\n".join(stderr_buf) or "(no stderr captured)"
            raise RuntimeError(
                f"ffmpeg exited with code {rc} after writing {i} frames. "
                f"stderr:\n{tail}"
            )

        # ---- Final summary: authoritative per-stage table ----
        total = sum(stage_times[k] for k in buckets_ordered) or 1e-9
        wallclock = time.time() - t0
        final_fps = i / max(wallclock, 1e-6)
        logline("ok", f"Upscaled {i} frames in {wallclock:.1f}s ({final_fps:.3f} fps).")

        summary_lines = [
            f"<b>Per-stage average (ms/frame) over {i} frames, {final_fps:.3f} fps wall:</b>"
        ]
        for k in buckets_ordered:
            v = stage_times[k]
            summary_lines.append(
                f"&nbsp;&nbsp;{k:14s}: {v/i*1000:8.1f} ms/frame  ({v/total*100:4.1f}%)"
            )
        # GPU util min/max across the run — the actual answer to
        # "is the GPU maxed out or idle?".
        if gpu_samples:
            utils = []
            for _, s in gpu_samples:
                try:
                    utils.append(int(s.split("gpu=")[1].split("%")[0]))
                except Exception:
                    pass
            if utils:
                summary_lines.append(
                    f"&nbsp;&nbsp;GPU util range: min={min(utils)}% "
                    f"max={max(utils)}% avg={sum(utils)//len(utils)}% "
                    f"across {len(utils)} samples"
                )
        logline("perf", "<br>".join(summary_lines).replace("\n", "<br>"))

        # 4. Mux original audio back in.
        stage_lbl.value = "<b>Muxing original audio…</b>"
        with_audio = out_dir / f"{stem}_upscaled.mp4"
        rc = subprocess.run(
            ["ffmpeg","-y","-hide_banner","-loglevel","error",
             "-i",str(video_only),"-i",str(in_path),
             "-map","0:v:0","-map","1:a:0?","-c:v","copy","-c:a","aac","-b:a","192k",
             "-shortest", str(with_audio)],
            capture_output=True, text=True,
        )
        if rc.returncode != 0:
            if rc.stderr:
                logline("warn", f"Audio mux failed: {rc.stderr.strip().splitlines()[-1] if rc.stderr.strip() else 'unknown'}")
            shutil.move(str(video_only), str(with_audio))
            logline("warn","Source had no audio track (or codec mismatch); output is silent.")
        else:
            try: os.remove(video_only)
            except OSError: pass
            logline("ok","Audio muxed back in.")

        final = with_audio

        # 5. Optional 1080p-height downscale.
        if cb_1080.value:
            stage_lbl.value = "<b>Downscaling to 1080p height (aspect-preserving)…</b>"
            down = out_dir / f"{stem}_upscaled_1080p.mp4"
            rc = subprocess.run(
                ["ffmpeg","-y","-hide_banner","-loglevel","error",
                 "-i",str(final),
                 "-vf","scale=-2:1080:flags=lanczos",
                 "-c:v","libx264","-preset","medium","-crf","17","-pix_fmt","yuv420p",
                 "-c:a","copy",
                 str(down)],
                capture_output=True, text=True,
            )
            if rc.returncode == 0:
                final = down
                logline("ok","Downscaled to 1080p height, aspect ratio preserved.")
            else:
                last = rc.stderr.strip().splitlines()[-1] if rc.stderr.strip() else "unknown"
                logline("warn", f"1080p downscale failed ({last}) — keeping full-res output.")

        MS["output_path"] = final
        stage_lbl.value = f"<b>Done.</b> Output: <code>{final.name}</code>"
        logline("ok", f"Ready for Step 4. File: <b>{final.name}</b> · {final.stat().st_size/1e6:.1f} MB.")

    start.on_click(start_processing)

    display(widgets.VBox([
        hdr, tier,
        s_revert, s_detail, s_sharpen, s_denoise, s_dehalo, s_deblur, s_recover,
        cb_1080,
        widgets.HTML("<hr style='border:none;border-top:1px solid #e2e8f0;margin:8px 0;'>"),
        start,
        stage_lbl,
        widgets.HBox([frame_prog, frame_pct]),
        log,
    ], layout=widgets.Layout(border="1px solid #e2e8f0", padding="14px", border_radius="10px")))


## Step 4 — Download

Click the button. Your browser downloads the result directly. No public/shareable link is generated — the file only exists on your machine and the Colab VM.

In [ ]:
#@title ⬇️ Step 4 — Download result { display-mode: "form" }
import ipywidgets as widgets
from IPython.display import display, HTML

MS = globals().setdefault("MOTIONSALT", {})
if not MS.get("output_path"):
    display(HTML('<div style="padding:10px;background:#fee2e2;color:#991b1b;border-radius:8px;font-family:sans-serif;">⚠️ Run Step 3 first (Configure & process).</div>'))
else:
    out = MS["output_path"]
    hdr = widgets.HTML(
        f'<div style="font-family:-apple-system,Segoe UI,sans-serif;font-weight:700;font-size:16px;color:#0f172a;">Your file is ready</div>'
        f'<div style="font-family:-apple-system,Segoe UI,sans-serif;font-size:12.5px;color:#475569;margin-bottom:10px;"><code>{out.name}</code> · {out.stat().st_size/1e6:.1f} MB</div>'
    )
    btn = widgets.Button(description="⬇️ Download Result", button_style="primary",
                         layout=widgets.Layout(width="240px", height="46px"))
    status = widgets.HTML("")

    def on_click(_):
        from google.colab import files
        btn.disabled = True
        status.value = '<div style="font-family:sans-serif;color:#475569;font-size:12.5px;">Preparing browser download…</div>'
        files.download(str(out))
        status.value = '<div style="padding:10px;background:#dcfce7;color:#166534;border-radius:8px;font-family:sans-serif;">✅ Download started in your browser.</div>'
        btn.disabled = False

    btn.on_click(on_click)
    display(widgets.VBox([hdr, btn, status],
                         layout=widgets.Layout(border="1px solid #e2e8f0", padding="14px", border_radius="10px")))


---

<div align="center" style="font-family:-apple-system,Segoe UI,sans-serif;color:#64748b;font-size:12px;padding:10px;">
MOTIONSALT Upscaler · MIT-licensed wrapper · powered by
<a href="https://github.com/the-database/mpv-upscale-2x_animejanai">AnimeJaNai V3</a> and
<a href="https://github.com/xinntao/Real-ESRGAN">Real-ESRGAN AnimeVideo v3</a>.<br>
Source: <a href="https://github.com/motionssalt/upscale">github.com/motionssalt/upscale</a>
</div>